# W2.3 — Audit leakage độc lập

**Chủ sở hữu:** B. Phạm vi tuần 2–4, độc lập branch A.

**Input:** edges.npz + manifest.json từ W2.2; đọc lại từ đĩa.

Audit kiểm tra cả test pairs để kiểm tra giao tập, tuyệt đối không dự đoán test.
Đối chứng phải chủ động chèn lỗi; assertion không phát hiện lỗi sẽ làm notebook thất bại.

In [ ]:
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p/'configs').is_dir() and (p/'notebooks').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Hãy mở Jupyter từ repo hoặc thư mục con có configs/ và notebooks/.')

In [ ]:
get_ipython().run_line_magic('run', '"' + (PROJECT_ROOT/'notebooks/shared/runtime.ipynb').as_posix() + '"')
get_ipython().run_line_magic('run', '"' + (PROJECT_ROOT/'notebooks/shared/graph_ops.ipynb').as_posix() + '"')

D:\gnn-node-link-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


D:\gnn-node-link-prediction\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
config = read_json(PROJECT_ROOT/'configs/data_protocol.json')
print(config)

{'dataset': 'Cora', 'public_node_split': 'public', 'seed': 42, 'undirected': True, 'val_ratio': 0.05, 'test_ratio': 0.1, 'negative_ratio': 1, 'split_id': 'cora_lp_v1', 'self_loops': 'exclude_from_pairs_add_once_per_node_in_layer', 'rounding': 'floor_val_floor_test_train_remainder'}


In [ ]:
data_config = read_json(PROJECT_ROOT/'configs/data_protocol.json')
data = load_cora(PROJECT_ROOT,data_config)
full_pairs = canonicalize(data.edge_index.numpy(),data.num_nodes)
fingerprints = {'features':array_hash(data.x.numpy()),'full_pairs':array_hash(full_pairs)}

In [ ]:
split_dir = PROJECT_ROOT/'artifacts/splits'/data_config['split_id']
manifest = read_json(split_dir/'manifest.json')
if not (split_dir/'edges.npz').exists():
    raise FileNotFoundError('Thiếu edges.npz: chạy W2.2 trước.')
with np.load(split_dir/'edges.npz',allow_pickle=False) as archive:
    arrays = {k:archive[k] for k in archive.files}
assert manifest['input_fingerprints'] == fingerprints, 'Dataset fingerprint thay đổi'
assert manifest['config'] == data_config, 'Protocol thay đổi: dùng split ID mới'
assert mapping_hash(arrays) == manifest['split_hash'], 'Split hash sai'
assert {k:array_hash(v) for k,v in arrays.items()} == manifest['array_hashes']
assert manifest['counts'] == {k:v.shape[1] for k,v in arrays.items()}
assert manifest['num_nodes'] == data.num_nodes
train_adjacency = bidirectional(arrays['train_pos'],data.num_nodes)

In [ ]:
checks = audit_split(arrays,full_pairs,data.num_nodes,train_adjacency)
regenerated = make_split(full_pairs,data.num_nodes,data_config)
assert all(np.array_equal(arrays[k],regenerated[k]) for k in arrays)
checks['seed_reproducibility'] = {'status':'passed','split_hash':mapping_hash(regenerated)}
toy_full = np.array([[0,0,1,2,3,4,5,6,7,8],[1,2,2,3,4,5,6,7,8,9]],dtype=np.int64)
toy_config = dict(data_config,val_ratio=.2,test_ratio=.2)
toy_arrays = make_split(toy_full,12,toy_config)
toy_adj = bidirectional(toy_arrays['train_pos'],12)
audit_split(toy_arrays,toy_full,12,toy_adj)
leaky_adj = np.concatenate([toy_adj,toy_arrays['val_pos'][::-1,:1]],axis=1)
checks['detect_reverse_validation_in_adjacency'] = expect_error(lambda:audit_split(toy_arrays,toy_full,12,leaky_adj))
corrupt = {k:v.copy() for k,v in toy_arrays.items()}
corrupt['train_neg'][:,0] = toy_arrays['test_pos'][:,0]
checks['detect_negative_heldout_positive'] = expect_error(lambda:audit_split(corrupt,toy_full,12,toy_adj))
run = new_run(PROJECT_ROOT,'w2_audit',config)
audit = publish_result(PROJECT_ROOT,run,'results/week02/leakage_audit.json',
                       {'status':'passed','checks':checks,'input_fingerprints':fingerprints},config,2,'W2.3','audit','Cora',manifest)
print('Audit:',audit['status'], 'checks:',len(checks), 'run:',run.name)

Audit: passed checks: 13 run: 20260928T070114_w2_audit_2c83ad20


## Kết luận và tự kiểm tra

Chỉ các assertion chạy thành công mới tạo kết quả bàn giao. Output bên trên chứa đường dẫn/run ID để truy vết. B chưa được đánh dấu đã tự review.

1. Cạnh validation đảo hướng có gây leakage không?
2. Tại sao audit có thể đọc test pairs?
3. Fingerprint khác thì baseline cần làm gì?